# Mean-Reversion / Index-Support Screener + Bracket Backtester

**What it does (two stages):**

**Stage 1 — the screen.** For a large-cap universe it measures each stock's *natural behavior* and
ranks the ones that fit the profile Jake defined:
- **Typical daily range 2-3%** (median true-range %) — moves enough to trade, not a corpse
- **Mean-reverting** — lag-1 return autocorrelation < 0, Hurst exponent < 0.5, short reversion half-life
- **Index-supported** — high R-squared to the S&P (SPY): moves *with* the mechanical passive bid,
  not on its own lone-wolf catalysts
- **Large-cap** — the universe is S&P 100 by construction
- **No crash history** — worst single day better than -15% over the lookback

**Stage 2 — the backtest.** Takes a name and runs the bracket idea (buy, sell at +X%, rebuy at -X%,
repeat) against just buying and holding — so you SEE whether the consistency is the tradeable kind
or whether the mechanical bid just grinds it up past your sell.

---
### Honest limits (read once)
- **Behavior is NOT stationary.** Every metric here is *past tense* — a stock that mean-reverted can
  start trending with no memo. The screen reduces the odds of picking a trender; it can't eliminate them.
  (MU almost certainly screened as a clean mean-reverter at \$400, right before its 4x.)
- **The crash filter is odds, not armor.** 'Never dropped 15%' is not 'won't.' Size for the tail anyway.
- **The backtest is single-lot and ignores costs, slippage, and taxes.** Real bracket trading triggers
  wash sales constantly (gains taxed, losses deferred) — after-tax is *worse* than shown. Multi-lot
  averaging-down (your 'repeat' on every dip) would AMPLIFY the downside beyond what's plotted here.
- **yfinance** daily data; a handful of tickers may fail to download (handled, just skipped).


In [ ]:
# If needed on a fresh runtime, uncomment:
# !pip install yfinance --quiet
import numpy as np, pandas as pd, yfinance as yf
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)


## CONFIG — all the knobs live here


In [ ]:
# ---- data window ----
LOOKBACK = '2y'          # history to measure behavior over
INDEX_ETF = 'SPY'        # benchmark for index-support (R-squared)

# ---- the 'ideal profile' thresholds (Stage 1 flags) ----
RANGE_LO, RANGE_HI = 1.5, 3.5     # median daily true-range %, target band ('2-3% ATR')
AUTOCORR_MAX       = 0.0          # lag-1 autocorr must be < this (negative = mean-reverting)
HURST_MAX          = 0.50         # Hurst < 0.5 = mean-reverting
HL_LO, HL_HI       = 2, 40        # reversion half-life in days (tradeable band)
R2_MIN             = 0.35         # R-squared to SPY (index-supported)
CRASH_MAX          = -15.0        # worst single-day % must be BETTER (higher) than this

# ---- Stage 2 backtest defaults ----
BT_SELL_PCT = 0.025      # sell bracket: +2.5%
BT_BUY_PCT  = 0.025      # rebuy bracket: -2.5% from the sell
BT_CASH     = 10000

# ---- universe: S&P 100 (large-cap by construction) ----
UNIVERSE = ['AAPL','MSFT','NVDA','AMZN','GOOGL','GOOG','META','BRK-B','LLY','AVGO','TSLA','JPM',
  'V','XOM','UNH','MA','JNJ','PG','HD','COST','ORCL','MRK','ABBV','CVX','KO','PEP','ADBE','WMT',
  'BAC','CRM','MCD','TMO','CSCO','ACN','ABT','LIN','DHR','INTC','AMD','QCOM','TXN','PM','WFC',
  'DIS','CAT','VZ','INTU','IBM','GE','NEE','AMGN','NOW','UNP','HON','LOW','SPGI','GS','ISRG',
  'BKNG','AXP','MS','RTX','PFE','BLK','C','SCHW','T','CVS','DE','LMT','BA','MDT','ELV','SBUX',
  'GILD','ADP','MDLZ','CB','MMC','AMT','TJX','SO','BMY','DUK','MO','CI','NKE','PLD','USB','PNC',
  'CL','COP','EMR','FDX','GD','ITW','MET','KHC','F','GM']


## Behavior metrics (the math the screen runs)


In [ ]:
def typical_range_pct(high, low, close):
    prev = close.shift(1)
    tr = pd.concat([(high-low),(high-prev).abs(),(low-prev).abs()], axis=1).max(axis=1)
    return (tr/close*100).median()

def lag1_autocorr(returns):
    r = returns.dropna()
    return r.autocorr(lag=1) if len(r) > 30 else np.nan

def hurst_exponent(prices):
    p = np.asarray(prices, dtype=float); p = p[~np.isnan(p)]
    if len(p) < 120: return np.nan
    logp = np.log(p)
    lags = range(2, 50)
    tau = []
    for lag in lags:
        d = logp[lag:] - logp[:-lag]
        s = np.std(d)
        tau.append(s if s > 0 else 1e-10)
    return float(np.polyfit(np.log(list(lags)), np.log(tau), 1)[0])

def half_life(prices):
    s = pd.Series(prices).dropna()
    if len(s) < 60: return np.nan
    lag = s.shift(1); delta = (s - lag).dropna(); lag = lag.loc[delta.index]
    beta = np.polyfit(lag.values, delta.values, 1)[0]
    return np.nan if beta >= 0 else float(-np.log(2)/beta)

def r2_to_index(stock_ret, idx_ret):
    df = pd.concat([stock_ret, idx_ret], axis=1).dropna()
    if len(df) < 60: return np.nan
    c = np.corrcoef(df.iloc[:,0], df.iloc[:,1])[0,1]
    return float(c**2)

def worst_day_pct(returns):
    r = returns.dropna()
    return float(r.min()*100) if len(r) else np.nan


## Download the universe


In [ ]:
tickers = sorted(set(UNIVERSE + [INDEX_ETF]))
print(f'Downloading {len(tickers)} tickers ({LOOKBACK}) ...')
raw = yf.download(tickers, period=LOOKBACK, auto_adjust=True, progress=False, group_by='column')

# normalize: want raw['Close'] etc. as DataFrames of ticker columns
def field(df, name):
    if isinstance(df.columns, pd.MultiIndex):
        return df[name]
    return df[[name]]   # single-ticker fallback

close_all = field(raw, 'Close'); high_all = field(raw, 'High'); low_all = field(raw, 'Low')
idx_ret = close_all[INDEX_ETF].pct_change()
print('Downloaded. Close shape:', close_all.shape)


## Stage 1 — score every name, rank the ones that fit the profile


In [ ]:
rows = []
for t in UNIVERSE:
    try:
        if t not in close_all.columns: continue
        c = close_all[t].dropna(); h = high_all[t]; l = low_all[t]
        if len(c) < 150: continue
        ret = c.pct_change()
        rng = typical_range_pct(h, l, c)
        ac  = lag1_autocorr(ret)
        hu  = hurst_exponent(c.values)
        hl  = half_life(c.values)
        r2  = r2_to_index(ret, idx_ret)
        wd  = worst_day_pct(ret)
        rows.append(dict(ticker=t, range_pct=rng, autocorr=ac, hurst=hu,
                         half_life=hl, r2_spy=r2, worst_day=wd))
    except Exception as e:
        print('skip', t, e)

df = pd.DataFrame(rows).set_index('ticker')

# per-criterion PASS flags
df['f_range'] = df['range_pct'].between(RANGE_LO, RANGE_HI)
df['f_revert'] = (df['autocorr'] < AUTOCORR_MAX) & (df['hurst'] < HURST_MAX)
df['f_halflife'] = df['half_life'].between(HL_LO, HL_HI)
df['f_index'] = df['r2_spy'] > R2_MIN
df['f_nocrash'] = df['worst_day'] > CRASH_MAX
flag_cols = ['f_range','f_revert','f_halflife','f_index','f_nocrash']
df['passes'] = df[flag_cols].sum(axis=1)

# rank: most criteria passed, then closest to 2.5% range, then most index-supported
df['range_fit'] = (df['range_pct'] - 2.5).abs()
ranked = df.sort_values(['passes','f_revert','r2_spy','range_fit'],
                        ascending=[False,False,False,True])

show = ['range_pct','autocorr','hurst','half_life','r2_spy','worst_day','passes']
print('=== FULL PROFILE (all 5 criteria must pass = 5) ===')
print(ranked[show].round(3).to_string())


### How to read Stage 1
- **passes = 5** → fits every criterion: 2-3% range, mean-reverting, tradeable half-life, index-supported,
  no crash. These are the candidates your idea is *designed* for.
- **autocorr** negative and **hurst** below 0.5 = genuinely mean-reverting (the two must agree).
- **r2_spy** high = it rides the mechanical passive bid (your index-support insight) rather than its own
  lone-wolf news. Low R-squared names (MU-like) are where the 28% surprise days live.
- **half_life** = days to revert halfway. ~2-40 is tradeable; very long = you'll wait forever.

Now Stage 2: take the top name (or set TICKER yourself) and see if the bracket actually beats holding it.


## Stage 2 — bracket vs buy-and-hold (does the consistency actually pay?)


In [ ]:
def max_dd(equity):
    roll = equity.cummax()
    return float((equity/roll - 1).min()*100)

def backtest_bracket(close, sell_pct, buy_pct, start_cash=10000):
    '''Single-lot bracket: start invested, sell at +sell_pct, rebuy at -buy_pct from the sell price,
    repeat. Single-lot on purpose (no infinite averaging-down) so it is CONSERVATIVE vs the real
    multi-lot version, which would be worse on the downside.'''
    c = close.dropna(); px = c.values; dates = c.index
    cash = 0.0; shares = start_cash/px[0]; entry = px[0]; buy_trigger = None
    equity = []; round_trips = 0; days_in_cash = 0
    for p in px:
        if shares > 0:
            if p >= entry*(1+sell_pct):
                cash = shares*p; shares = 0.0; buy_trigger = p*(1-buy_pct); round_trips += 1
        else:
            days_in_cash += 1
            if buy_trigger is not None and p <= buy_trigger:
                shares = cash/p; cash = 0.0; entry = p
        equity.append(cash + shares*p)
    eq = pd.Series(equity, index=dates)
    hold = pd.Series(px/px[0]*start_cash, index=dates)
    return eq, hold, round_trips, days_in_cash

def run(ticker, sell_pct=BT_SELL_PCT, buy_pct=BT_BUY_PCT, start_cash=BT_CASH):
    c = close_all[ticker].dropna()
    eq, hold, rt, dic = backtest_bracket(c, sell_pct, buy_pct, start_cash)
    br_ret = (eq.iloc[-1]/start_cash - 1)*100
    ho_ret = (hold.iloc[-1]/start_cash - 1)*100
    print(f'--- {ticker}  bracket +{sell_pct*100:.1f}% / -{buy_pct*100:.1f}% over {LOOKBACK} ---')
    print(f'  Bracket total return : {br_ret:7.1f}%   (max DD {max_dd(eq):6.1f}%)')
    print(f'  Buy & hold return    : {ho_ret:7.1f}%   (max DD {max_dd(hold):6.1f}%)')
    print(f'  Bracket - Hold       : {br_ret-ho_ret:7.1f}%   <-- the cost/benefit of trading it')
    print(f'  Round-trips: {rt}   Days parked in cash: {dic}/{len(c)}')
    plt.figure(figsize=(10,5))
    plt.plot(hold.index, hold.values, label=f'Buy & Hold ({ho_ret:+.0f}%)', lw=2)
    plt.plot(eq.index, eq.values, label=f'Bracket ({br_ret:+.0f}%)', lw=2)
    plt.title(f'{ticker}: bracket vs hold'); plt.legend(); plt.grid(alpha=0.3); plt.show()
    return eq, hold

# auto-pick the best-ranked full-profile name; or hard-set TICKER = 'KO' etc.
best = ranked.index[0]
TICKER = best
print('Top-ranked profile fit:', best, '| passes =', int(ranked.loc[best,'passes']))
_ = run(TICKER)


### Compare a mean-reverter vs a lone-wolf
Run it on the top profile-fit AND on a low-R-squared / high-vol name (e.g. a semi) to *see* the
difference: on a true range-bound name the bracket may keep pace; on a trender it caps you at +2.5%
while hold runs away (equity flatlines in cash) or it rebuys into a slide. Set the two tickers below.


In [ ]:
for t in [ranked.index[0], ranked.index[1], 'MU', 'NVDA']:
    if t in close_all.columns:
        try: run(t)
        except Exception as e: print('skip', t, e)


---
### The read once you run it
If the bracket beats hold on the top names, the *consistency is tradeable* — build around it, but keep
the live-behavior gate (re-check autocorr/Hurst on a rolling window; stand down when it flips positive)
and size for the tail the crash filter can't remove. If hold wins (the usual result on index-supported
names, because the passive bid *trends* them up past your sell), then the mechanical consistency you
correctly spotted is a **reason to OWN them**, not to scalp them. Either way — the market told us, with
numbers.
